# 01 · Features

Le os dados brutos e produz a tabela de features `data/processed/features.parquet`.

Implementa a especificacao fechada em `docs/features/decisoes.md` (etapas E0-E6).
A *mesma* rotina computa as features para o historico (`split="train"`) e para os
72 jogos da Copa 2026 (`split="predict"`), garantindo consistencia treino-previsao.

### Contrato
**Entrada:** `data/raw/historical-results.csv`, `data/raw/ranking.csv`, `data/raw/matches-schedule.csv`

**Saida:** `data/processed/features.parquet` (uma linha por partida; colunas na ordem da secao 10 da spec)

### Regras invioláveis
- **Zero vazamento temporal:** cada feature usa apenas informacao conhecida *antes*
  do apito inicial. Elo grava o valor pre-jogo; janelas moveis usam `shift(1)`;
  ranking via `merge_asof` estritamente antes da data.
- **Nunca ler o CSV de resultados reais de 2026** (o gabarito): as previsões dos
  72 jogos saem do estado pré-torneio, sem realimentar resultados do Mundial.
- **Reprodutivel:** `SEED=42`, operacoes deterministicas e vetorizadas.

In [19]:
# Setup: resolve a raiz do repositório e os caminhos de dados.
from pathlib import Path


def find_root(start: Path | None = None) -> Path:
    base = (start or Path.cwd()).resolve()
    for cand in [base, *base.parents]:
        if (cand / "data" / "raw").is_dir() and (cand / "pyproject.toml").is_file():
            return cand
    raise RuntimeError("Raiz do repositório não encontrada a partir de " + str(base))


ROOT = find_root()
RAW = ROOT / "data" / "raw"
PROCESSED = ROOT / "data" / "processed"
RESULTS = ROOT / "data" / "results"
MODELS = ROOT / "models"
print("ROOT:", ROOT)
PROCESSED.mkdir(parents=True, exist_ok=True)

ROOT: /Users/franklaercio/Workspace/paul-the-octopus


## E0 · Prep, ambiente e reconciliação

Garante `pyarrow` (necessário para gravar o Parquet), fixa a semente, carrega as
três fontes, aplica `canon()` (mapa único da skill `avaliar-previsoes`), deduplica
o confronto duplicado e monta a tabela de partidas unificada (histórico + 2026) em
ordem cronológica estável.

In [20]:
# Garante o pyarrow (engine do Parquet). Em ambientes já provisionados é no-op.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyarrow") is None:
    print("pyarrow ausente; instalando via requirements-dev.txt...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(ROOT / "requirements-dev.txt")],
        check=True,
    )
import pyarrow  # noqa: F401

print("pyarrow:", pyarrow.__version__)

pyarrow: 24.0.0


In [21]:
# Constantes fixas da spec (docs/features/decisoes.md - "cola rápida").
import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)

ELO_INICIAL = 1500.0
ELO_K = 40.0
ELO_HFA = 65.0  # vantagem de mando, só quando is_neutral == False
FORM_N = 5  # janela de forma (média simples, shift(1))
REST_CAP = 30  # dias; saturação da fadiga
H2H_DEFAULT = 0.5  # winrate quando não há confronto prévio
HALFLIFE_ANOS = 8.0  # meia-vida do decaimento de recência no sample_weight
DATA_REF = pd.Timestamp("2026-06-11")  # 1º dia da Copa; NÃO derivado do gabarito
HOSTS = {"usa", "canada", "mexico"}  # anfitriões (canônicos): 9 jogos não-neutros

OUTCOMES = ("home", "draw", "away")  # ordem ordinal (casa com a skill de avaliação)

In [22]:
# Fonte única de nomes: reutiliza DEFAULT_ALIASES/_norm da skill avaliar-previsoes.
# Fallback inline replica o mapa literalmente caso o import falhe (mesma verdade).
import sys
import unicodedata

_SKILL = ROOT / ".claude" / "skills" / "avaliar-previsoes" / "scripts"
if str(_SKILL) not in sys.path:
    sys.path.insert(0, str(_SKILL))

try:
    from score_predictions import DEFAULT_ALIASES, _norm
except ImportError:  # pragma: no cover - fallback se a skill não estiver acessível
    DEFAULT_ALIASES = {
        "united states": "usa",
        "united states of america": "usa",
        "korea republic": "south korea",
        "korea dpr": "north korea",
        "ir iran": "iran",
        "iran islamic republic of": "iran",
        "cote d'ivoire": "ivory coast",
        "cote d ivoire": "ivory coast",
        "czechia": "czech republic",
        "turkiye": "turkey",
        "cabo verde": "cape verde",
        "china pr": "china",
        "the netherlands": "netherlands",
        "bosnia": "bosnia and herzegovina",
    }

    def _norm(name: object) -> str:
        s = unicodedata.normalize("NFKD", str(name)).encode("ascii", "ignore").decode()
        return " ".join(s.lower().split())


def canon(s: pd.Series) -> pd.Series:
    """Aplica a forma canônica (mapa da skill) a uma Series de nomes de seleção."""
    return s.map(lambda x: DEFAULT_ALIASES.get(_norm(x), _norm(x)))

In [23]:
# Carga das fontes (apenas histórico, ranking e calendário). O gabarito de 2026
# (resultados reais) é deliberadamente NÃO carregado aqui.
historico = pd.read_csv(RAW / "historical-results.csv", parse_dates=["date"])
ranking = pd.read_csv(RAW / "ranking.csv", parse_dates=["rank_date"])
calendario = pd.read_csv(RAW / "matches-schedule.csv")

historico["home"] = canon(historico["home_team"])
historico["away"] = canon(historico["away_team"])

# Dedupe: o par (date, home, away) tem 1 confronto duplicado com placares
# divergentes (1974 Tahiti x New Caledonia). keep=False remove AMBAS as linhas.
antes = len(historico)
historico = historico[
    ~historico.duplicated(subset=["date", "home", "away"], keep=False)
].copy()
print(f"histórico: {antes} -> {len(historico)} linhas (removidas {antes - len(historico)} do par duplicado)")

histórico: 49402 -> 49400 linhas (removidas 2 do par duplicado)


In [24]:
# Calendário 2026: canonizar, número do jogo, e is_neutral por proxy de anfitrião.
calendario["home"] = canon(calendario["home"])
calendario["away"] = canon(calendario["away"])
calendario["match_no"] = calendario["match"].astype("Int64")
# Proxy de mando (limitação registrada): jogo é não-neutro só quando o home nominal
# é anfitrião (usa/canada/mexico); todos os demais são neutros. -> 9 não-neutros.
calendario["is_neutral"] = ~calendario["home"].isin(HOSTS)
print(
    "calendário:", len(calendario), "jogos |",
    int((~calendario["is_neutral"]).sum()), "não-neutros (anfitriões) |",
    int(calendario["is_neutral"].sum()), "neutros",
)

calendário: 72 jogos | 9 não-neutros (anfitriões) | 63 neutros


In [25]:
# Tabela de partidas unificada (home-oriented, uma linha por jogo), em ordem
# cronológica estável (date, home, away). As linhas de 2026 entram na varredura,
# mas NÃO realimentam Elo/forma/H2H (são o estado pré-torneio) - garantido adiante.
hist_cols = historico[
    ["date", "home", "away", "home_score", "away_score", "neutral", "tournament"]
].rename(columns={"neutral": "is_neutral"})
hist_cols = hist_cols.assign(
    split="train", match_no=pd.array([pd.NA] * len(hist_cols), dtype="Int64")
)

pred_cols = calendario[["date", "home", "away", "is_neutral", "match_no"]].copy()
pred_cols["date"] = pd.to_datetime(pred_cols["date"], format="%d/%m/%Y")
pred_cols = pred_cols.assign(
    home_score=pd.NA, away_score=pd.NA, tournament="FIFA World Cup", split="predict"
)

matches = pd.concat([hist_cols, pred_cols], ignore_index=True)
matches["date"] = matches["date"].astype("datetime64[ns]")
matches = matches.sort_values(["date", "home", "away"], kind="mergesort").reset_index(drop=True)
matches["mid"] = np.arange(len(matches))  # id estável por partida
print(
    "matches:", matches.shape,
    "| train:", int((matches.split == "train").sum()),
    "| predict:", int((matches.split == "predict").sum()),
)

matches: (49472, 10) | train: 49400 | predict: 72


## E1 · Núcleo de força: Elo pré-jogo + ranking FIFA *as-of*

São o sinal mais forte da EDA. O Elo varre **todo** o histórico em ordem
cronológica, gravando o valor pré-jogo e atualizando depois (HFA só em jogos
não-neutros; multiplicador de margem da spec). As linhas `predict` **não**
atualizam o Elo (sem vazamento): o Elo de 2026 é o snapshot pré-torneio, igual nos
3 jogos de cada seleção.

O ranking entra via `merge_asof` *backward* (a publicação estritamente anterior à
data; `allow_exact_matches=False`). Usamos `rank`/`rank_diff`/`points_diff` —
**não** o `total_points` absoluto (escala incomparável entre eras).

In [26]:
# Elo pré-jogo via varredura cronológica única (vetorizada em arrays numpy).
# Grava o rating ANTES do jogo; atualiza DEPOIS apenas para linhas train.
def compute_elo(df: pd.DataFrame) -> pd.DataFrame:
    n = len(df)
    elo_home = np.empty(n, dtype=float)
    elo_away = np.empty(n, dtype=float)
    home = df["home"].to_numpy()
    away = df["away"].to_numpy()
    hs = pd.to_numeric(df["home_score"]).to_numpy()
    as_ = pd.to_numeric(df["away_score"]).to_numpy()
    neutral = df["is_neutral"].to_numpy()
    split = df["split"].to_numpy()

    ratings: dict[str, float] = {}
    for i in range(n):
        rh = ratings.get(home[i], ELO_INICIAL)
        ra = ratings.get(away[i], ELO_INICIAL)
        elo_home[i] = rh
        elo_away[i] = ra
        if split[i] != "train":
            continue  # predict: snapshot pré-torneio, nunca realimenta
        dr = (rh + (0.0 if neutral[i] else ELO_HFA)) - ra
        e_home = 1.0 / (1.0 + 10.0 ** (-dr / 400.0))
        gd = abs(int(hs[i]) - int(as_[i]))
        if gd <= 1:
            g = 1.0
        elif gd == 2:
            g = 1.5
        else:
            g = (11 + gd) / 8.0
        s_home = 1.0 if hs[i] > as_[i] else (0.5 if hs[i] == as_[i] else 0.0)
        ratings[home[i]] = rh + ELO_K * g * (s_home - e_home)
        ratings[away[i]] = ra + ELO_K * g * ((1.0 - s_home) - (1.0 - e_home))

    out = df.copy()
    out["elo_home"] = elo_home
    out["elo_away"] = elo_away
    out["elo_diff"] = elo_home - elo_away  # sem HFA (o mando vai em is_neutral)
    return out


matches = compute_elo(matches)
print("elo_home faixa:", round(matches["elo_home"].min(), 1), "..", round(matches["elo_home"].max(), 1))
# invariante anti-vazamento: cada seleção tem 1 só Elo nos seus jogos de 2026.
_p = matches[matches.split == "predict"]
assert _p.groupby("home")["elo_home"].nunique().max() == 1, "Elo de 2026 deveria ser snapshot"

elo_home faixa: 764.6 .. 2266.2


In [27]:
# Ranking FIFA as-of: última publicação estritamente antes da data da partida.
# Descarta os 21 rank nulos ANTES do merge_asof (não propagar). NÃO usamos
# total_points absoluto como feature (só points_diff).
rk = ranking.copy()
rk["k"] = canon(rk["country_full"])
rk["rank_date"] = rk["rank_date"].astype("datetime64[ns]")  # alinhar dtype com matches.date
rk = rk.dropna(subset=["rank"]).sort_values("rank_date", kind="mergesort").reset_index(drop=True)


def rank_asof(side: str) -> pd.DataFrame:
    left = (
        matches[["mid", "date", side]]
        .rename(columns={side: "k"})
        .sort_values("date", kind="mergesort")
    )
    right = rk[["rank_date", "k", "rank", "total_points", "confederation"]].sort_values(
        "rank_date", kind="mergesort"
    )
    merged = pd.merge_asof(
        left,
        right,
        left_on="date",
        right_on="rank_date",
        by="k",
        direction="backward",
        allow_exact_matches=False,
    )
    return merged.set_index("mid")


asof_home = rank_asof("home")
asof_away = rank_asof("away")

matches = matches.set_index("mid")
matches["rank_home"] = asof_home["rank"]
matches["rank_away"] = asof_away["rank"]
matches["rank_diff"] = matches["rank_away"] - matches["rank_home"]  # + = mandante melhor
matches["points_diff"] = asof_home["total_points"] - asof_away["total_points"]

# Confederação: as-of quando houver; senão a última conhecida da seleção; senão
# "unknown" (seleções extintas/não-FIFA do histórico - nenhuma em 2026).
conf_last = rk.sort_values("rank_date").groupby("k")["confederation"].last()
matches["confed_home"] = (
    asof_home["confederation"].fillna(matches["home"].map(conf_last)).fillna("unknown")
)
matches["confed_away"] = (
    asof_away["confederation"].fillna(matches["away"].map(conf_last)).fillna("unknown")
)
matches = matches.reset_index()

print("rank_home NaN (train, pré-1993):", int(matches.loc[matches.split == "train", "rank_home"].isna().sum()))
print("rank_home NaN (predict, deve ser 0):", int(matches.loc[matches.split == "predict", "rank_home"].isna().sum()))

rank_home NaN (train, pré-1993): 20479
rank_home NaN (predict, deve ser 0): 0


## E2 · Mando / campo neutro

`is_neutral` já foi definido na E0 (histórico = coluna `neutral`; 2026 = proxy de
anfitrião) e governa o HFA do Elo. Nada a derivar aqui além de confirmar a
coerência — o modelo aprende o peso de `is_neutral`; não criamos `home_advantage`
numérica.

> **Limitação registrada:** o calendário 2026 não traz sede/cidade; o proxy assume
> que anfitriões jogam em casa e os demais em sede neutra. É a melhor aproximação
> sem a coluna de sede e sem tocar no gabarito.

In [28]:
# is_neutral já consolidado em E0; confirmação de coerência.
assert matches["is_neutral"].notna().all(), "is_neutral não pode ter NaN"
print("is_neutral (histórico, média):", round(matches.loc[matches.split == "train", "is_neutral"].mean(), 3))
print("is_neutral (2026, não-neutros):", int((~matches.loc[matches.split == "predict", "is_neutral"]).sum()))

is_neutral (histórico, média): 0.264
is_neutral (2026, não-neutros): 9


## E3 · Forma recente (ofensiva/defensiva)

Janela de 5 jogos com `shift(1)`, calculada na **tabela longa** (uma linha por
seleção-jogo), para a forma considerar todos os jogos da seleção (casa e fora).
`form_gf`/`form_ga` já são a força ofensiva/defensiva (não criamos features extras).

A forma usa **apenas jogos com resultado** (histórico): para as linhas `predict` é
o snapshot do último estado histórico de cada seleção (igual nos 3 jogos de 2026),
nunca os próprios jogos do Mundial.

In [29]:
# Forma na tabela longa, só sobre jogos com resultado (train). Para train,
# shift(1) + rolling(5); para predict, snapshot = rolling(5) no fim da série
# histórica de cada seleção (sem shift, pois é o estado conhecido pré-torneio).
def _long_train(df: pd.DataFrame) -> pd.DataFrame:
    t = df[df.split == "train"]
    home_rows = t[["mid", "date", "home", "home_score", "away_score"]].rename(
        columns={"home": "team", "home_score": "gf", "away_score": "ga"}
    )
    home_rows["side"] = "home"
    away_rows = t[["mid", "date", "away", "away_score", "home_score"]].rename(
        columns={"away": "team", "away_score": "gf", "home_score": "ga"}
    )
    away_rows["side"] = "away"
    long = pd.concat([home_rows, away_rows], ignore_index=True)
    long["gf"] = pd.to_numeric(long["gf"])
    long["ga"] = pd.to_numeric(long["ga"])
    long = long.sort_values(["team", "date", "mid"], kind="mergesort").reset_index(drop=True)
    long["points"] = np.select([long.gf > long.ga, long.gf == long.ga], [3.0, 1.0], default=0.0)
    return long


long_form = _long_train(matches)
grp = long_form.groupby("team", sort=False)
# train: média móvel deslocada (não enxerga o próprio jogo)
for col in ("points", "gf", "ga"):
    long_form[f"form_{col}"] = grp[col].transform(
        lambda s: s.shift(1).rolling(FORM_N, min_periods=1).mean()
    )
# snapshot pré-torneio (sem shift): último valor da média móvel por seleção
snap = long_form.copy()
for col in ("points", "gf", "ga"):
    snap[f"snap_{col}"] = grp[col].transform(lambda s: s.rolling(FORM_N, min_periods=1).mean())
snapshot = snap.groupby("team").tail(1).set_index("team")[["snap_points", "snap_gf", "snap_ga"]]

# mapear forma de train de volta para os lados home/away por mid
fh = long_form[long_form.side == "home"].set_index("mid")
fa = long_form[long_form.side == "away"].set_index("mid")
matches = matches.set_index("mid")
rename = {"points": "pts", "gf": "gf", "ga": "ga"}
for col, short in rename.items():
    matches[f"form_{short}_home"] = fh[f"form_{col}"]
    matches[f"form_{short}_away"] = fa[f"form_{col}"]
matches = matches.reset_index()

# preencher predict com o snapshot pré-torneio
pmask = matches.split == "predict"
for col, short in rename.items():
    matches.loc[pmask, f"form_{short}_home"] = matches.loc[pmask, "home"].map(snapshot[f"snap_{col}"]).to_numpy()
    matches.loc[pmask, f"form_{short}_away"] = matches.loc[pmask, "away"].map(snapshot[f"snap_{col}"]).to_numpy()

print("predict form_pts_home NaN (deve ser 0):", int(matches.loc[pmask, "form_pts_home"].isna().sum()))

predict form_pts_home NaN (deve ser 0): 0


## E4 · Descanso / congestionamento

`rest_days = data − data do jogo anterior da mesma seleção`, capado em 30. É a
**única** feature que usa legitimamente datas de 2026 (o jogo anterior pode ser
outro jogo do calendário). Para o 1º jogo de cada seleção no Mundial, o "jogo
anterior" é o último jogo histórico (só **datas**, nunca resultados).

In [30]:
# rest_days na tabela longa COMPLETA (inclui datas de 2026, só datas - não vaza).
def _long_dates(df: pd.DataFrame) -> pd.DataFrame:
    home_rows = df[["mid", "date", "home"]].rename(columns={"home": "team"})
    home_rows["side"] = "home"
    away_rows = df[["mid", "date", "away"]].rename(columns={"away": "team"})
    away_rows["side"] = "away"
    return pd.concat([home_rows, away_rows], ignore_index=True)


long_dates = _long_dates(matches).sort_values(
    ["team", "date", "mid"], kind="mergesort"
).reset_index(drop=True)
prev_date = long_dates.groupby("team", sort=False)["date"].shift(1)
long_dates["rest"] = (long_dates["date"] - prev_date).dt.days.clip(upper=REST_CAP).astype(float)

matches = matches.set_index("mid")
matches["rest_days_home"] = long_dates[long_dates.side == "home"].set_index("mid")["rest"]
matches["rest_days_away"] = long_dates[long_dates.side == "away"].set_index("mid")["rest"]
matches = matches.reset_index()
print(
    "rest_days_home (predict) min/mean/max:",
    matches.loc[matches.split == "predict", "rest_days_home"].agg(["min", "mean", "max"]).round(1).to_dict(),
)

rest_days_home (predict) min/mean/max: {'min': 4.0, 'mean': 6.5, 'max': 16.0}


## E5 · Confronto direto (H2H), confederação e torneio

H2H **computado do próprio histórico** (não do CSV de ratios), orientado ao
mandante atual, com `shift(1)` por par não-ordenado para nunca contar o próprio
jogo. Confederação já veio do ranking *as-of* (E1). O `tournament` vira um
**ordinal de importância** (0–5) por baldes de domínio — o one-hot fica no `02`.

In [31]:
# H2H computado do histórico, orientado ao mandante ATUAL. Agrupa por par não
# ordenado (lo<hi); acumula vitórias de 'lo' com shift(1) (só jogos prévios).
matches = matches.sort_values(["date", "home", "away"], kind="mergesort").reset_index(drop=True)
lo = matches["home"].where(matches["home"] < matches["away"], matches["away"])
hi = matches["away"].where(matches["home"] < matches["away"], matches["home"])
matches["_lo"] = lo
matches["_hi"] = hi
home_is_lo = matches["home"] == matches["_lo"]

hs = pd.to_numeric(matches["home_score"])
as_ = pd.to_numeric(matches["away_score"])
# vitória de 'lo' neste jogo (NaN em predict -> não conta; só train tem resultado)
lo_win = np.where(
    matches.split == "train",
    np.where(home_is_lo, (hs > as_).astype(float), (as_ > hs).astype(float)),
    np.nan,
)
matches["_lo_win"] = lo_win

pair_grp = matches.groupby(["_lo", "_hi"], sort=False)
matches["h2h_games"] = pair_grp.cumcount().astype("int64")  # jogos prévios do par
lo_wins_prev = pair_grp["_lo_win"].transform(lambda s: s.shift(1).fillna(0).cumsum())
# winrate do mandante atual = vitórias do home / jogos prévios; 0.5 se não houver
denom = matches["h2h_games"].replace(0, np.nan)
home_wins_prev = np.where(home_is_lo, lo_wins_prev, matches["h2h_games"] - lo_wins_prev)
matches["h2h_winrate_home"] = np.where(
    matches["h2h_games"] > 0, home_wins_prev / denom, H2H_DEFAULT
)
matches["h2h_available"] = matches["h2h_games"] > 0
matches = matches.drop(columns=["_lo", "_hi", "_lo_win"])

print(
    "H2H 2026: jogos com histórico prévio:",
    int(matches.loc[matches.split == "predict", "h2h_available"].sum()), "/ 72",
)

H2H 2026: jogos com histórico prévio: 44 / 72


In [32]:
# Ordinal de importância do torneio (baldes de domínio; ordem real amistoso<...<Copa).
# Aplicar na ordem; primeira keyword que casa vence (case-insensitive).
import re


def tournament_tier(name: object) -> int:
    s = str(name).lower().strip()
    if re.fullmatch(r"fifa world cup", s):  # final, sem "qualification"
        return 5
    if ("qualification" not in s) and ("qualifying" not in s):
        if (
            re.search(r"euro\b", s)
            or re.search(r"copa am[eé]rica", s)
            or "african cup of nations" in s
            or "asian cup" in s
            or "gold cup" in s
            or "concacaf championship" in s
            or "oceania nations cup" in s
            or "confederations cup" in s
        ):
            return 4  # continental principal + Confederations
    if "world cup qualification" in s:
        return 3
    if ("qualification" in s) or ("qualifying" in s) or ("nations league" in s):
        return 2  # eliminatória continental + Nations League
    if re.fullmatch(r"friendly", s):
        return 1
    return 0  # outros


# Peso de importância correspondente ao tier (item 8a da spec).
TIER_TO_WEIGHT = {5: 3.0, 4: 2.5, 3: 2.0, 2: 1.5, 1: 1.0, 0: 0.8}

matches["tournament_tier"] = matches["tournament"].map(tournament_tier).astype("int64")
print("distribuição de tournament_tier:", matches["tournament_tier"].value_counts().sort_index().to_dict())

distribuição de tournament_tier: {0: 9650, 1: 18387, 2: 8236, 3: 8772, 4: 3391, 5: 1036}


## E6 · Montagem do `features.parquet`

Deriva o alvo 1X2 e o `sample_weight` (importância do torneio × decaimento de
recência, meia-vida 8 anos) — **só em `train`**. Monta o DataFrame final com dtypes
nullable, na ordem exata da §10 da spec, ordenado por `(split, date, home)`, e grava
em `data/processed/features.parquet`.

In [33]:
# Alvo 1X2 e placares (só train). target derivado do placar; rótulos casam com
# OUTCOMES = (home, draw, away). predict fica NaN.
tmask = matches.split == "train"
hs = pd.to_numeric(matches["home_score"])
as_ = pd.to_numeric(matches["away_score"])
matches["target_outcome"] = pd.array([pd.NA] * len(matches), dtype="string")
matches.loc[tmask, "target_outcome"] = np.select(
    [hs[tmask] > as_[tmask], hs[tmask] < as_[tmask]], ["home", "away"], default="draw"
)

# sample_weight = w_torneio * w_recencia (só train; predict NaN).
w_tour = matches["tournament_tier"].map(TIER_TO_WEIGHT).astype(float)
idade_anos = (DATA_REF - matches["date"]).dt.days / 365.25
w_rec = 0.5 ** (idade_anos / HALFLIFE_ANOS)
matches["sample_weight"] = np.where(tmask, (w_tour * w_rec).astype(float), np.nan)

print(
    "target (train):",
    matches.loc[tmask, "target_outcome"].value_counts().to_dict(),
)
print(
    "sample_weight (train) faixa:",
    round(float(matches.loc[tmask, "sample_weight"].min()), 4),
    "..",
    round(float(matches.loc[tmask, "sample_weight"].max()), 4),
)

target (train): {'home': 24210, 'away': 13957, 'draw': 11233}
sample_weight (train) faixa: 0.0 .. 2.416


In [34]:
# DataFrame final com dtypes nullable e ordem exata da §10 da spec.
features = pd.DataFrame()
features["split"] = matches["split"].astype("string")
features["match_no"] = matches["match_no"].astype("Int64")
features["date"] = matches["date"].dt.normalize().astype("datetime64[ns]")
features["home"] = matches["home"].astype("string")
features["away"] = matches["away"].astype("string")
features["is_neutral"] = matches["is_neutral"].astype("boolean")
features["confed_home"] = matches["confed_home"].astype("string")
features["confed_away"] = matches["confed_away"].astype("string")

float_cols = [
    "elo_home", "elo_away", "elo_diff",
    "rank_home", "rank_away", "rank_diff", "points_diff",
    "form_pts_home", "form_pts_away", "form_gf_home", "form_gf_away",
    "form_ga_home", "form_ga_away",
    "rest_days_home", "rest_days_away",
    "h2h_winrate_home", "sample_weight",
]
for col in float_cols:
    features[col] = matches[col].astype("float64")

features["h2h_games"] = matches["h2h_games"].astype("int64")
features["h2h_available"] = matches["h2h_available"].astype("boolean")
features["tournament_tier"] = matches["tournament_tier"].astype("int64")
features["target_outcome"] = matches["target_outcome"].astype("string")
features["home_score"] = pd.to_numeric(matches["home_score"], errors="coerce").astype("Int64")
features["away_score"] = pd.to_numeric(matches["away_score"], errors="coerce").astype("Int64")

COLUMN_ORDER = [
    "split", "match_no", "date", "home", "away", "is_neutral",
    "confed_home", "confed_away",
    "elo_home", "elo_away", "elo_diff",
    "rank_home", "rank_away", "rank_diff", "points_diff",
    "form_pts_home", "form_pts_away", "form_gf_home", "form_gf_away",
    "form_ga_home", "form_ga_away",
    "rest_days_home", "rest_days_away",
    "h2h_games", "h2h_winrate_home", "h2h_available",
    "tournament_tier", "target_outcome", "home_score", "away_score", "sample_weight",
]
features = features[COLUMN_ORDER]
features = features.sort_values(["split", "date", "home"], kind="mergesort").reset_index(drop=True)
print("features:", features.shape, "| colunas:", len(features.columns))

features: (49472, 31) | colunas: 31


In [35]:
# Guardas de contrato e anti-vazamento (falham alto se algo regredir).
n_predict = int((features.split == "predict").sum())
assert n_predict == 72, f"esperado 72 linhas predict, obtido {n_predict}"
assert sorted(features.loc[features.split == "predict", "match_no"].dropna().tolist()) == list(
    range(1, 73)
), "match_no de predict deve cobrir 1..72 sem buracos"

# alvo/placar/peso presentes sse split == train
for col in ("target_outcome", "home_score", "away_score", "sample_weight"):
    assert features.loc[features.split == "train", col].notna().all(), f"{col} ausente em train"
    assert features.loc[features.split == "predict", col].isna().all(), f"{col} presente em predict"

# sem NaN nas chaves
for col in ("split", "date", "home", "away", "is_neutral"):
    assert features[col].notna().all(), f"NaN em chave {col}"

# predict completo nas features de força/contexto
for col in ("elo_home", "elo_away", "rank_home", "rank_away", "form_pts_home", "rest_days_home"):
    assert features.loc[features.split == "predict", col].notna().all(), f"{col} NaN em predict"

# winrate em [0, 1]
assert features["h2h_winrate_home"].between(0.0, 1.0).all(), "h2h_winrate fora de [0,1]"
print("guardas de contrato e anti-vazamento: OK")

guardas de contrato e anti-vazamento: OK


In [36]:
# Gravar o artefato.
out = PROCESSED / "features.parquet"
features.to_parquet(out, index=False)
print("OK:", out, features.shape)
features.head(3)

OK: /Users/franklaercio/Workspace/paul-the-octopus/data/processed/features.parquet (49472, 31)


,split,match_no,date,home,away,is_neutral,confed_home,confed_away,elo_home,elo_away,...,rest_days_home,rest_days_away,h2h_games,h2h_winrate_home,h2h_available,tournament_tier,target_outcome,home_score,away_score,sample_weight
0,predict,1,2026-06-11,mexico,south africa,False,CONCACAF,CAF,1991.207859,1665.531272,...,7.0,5.0,4,0.500000,True,5,<NA>,<NA>,<NA>,NaN
1,predict,2,2026-06-11,south korea,czech republic,True,AFC,UEFA,1883.526049,1858.943408,...,8.0,7.0,3,0.666667,True,5,<NA>,<NA>,<NA>,NaN
2,predict,3,2026-06-12,canada,bosnia and herzegovina,False,CONCACAF,UEFA,1926.838760,1684.949285,...,7.0,6.0,0,0.500000,False,5,<NA>,<NA>,<NA>,NaN
